# Deriving the Logistic-Regression Gradient

Wiki reference for [the logistic gradient derivation](https://ml-viz-ruby.vercel.app/wiki/logistic-gradient-derivation).

**The idea in one sentence.** The gradient of binary cross-entropy w.r.t. the logits collapses
to the beautifully simple $\frac1n X^\top(\hat y - y)$ — because the two chain-rule pieces
($\partial L/\partial \hat y$ and the sigmoid derivative $\hat y(1-\hat y)$) **cancel** to leave
just the residual $\hat y - y$.

We derive and implement the gradient from scratch, **validate it against finite differences and
confirm the chain-rule cancellation**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — Sigmoid and cross-entropy from scratch

The key result: $\partial\mathcal{L}/\partial z_i = \hat{y}_i - y_i$.
The sigmoid derivative cancels with terms from the cross-entropy derivative.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def cross_entropy(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def logistic_gradient(X, y, w):
    """Gradient of binary cross-entropy: (1/n) X^T (y_hat - y)."""
    y_hat = sigmoid(X @ w)
    return (1 / len(y)) * X.T @ (y_hat - y)

# The 2-point worked example from the wiki
X = np.array([[1., 2.], [1., -1.]])
y = np.array([1., 0.])
w = np.zeros(2)

y_hat = sigmoid(X @ w)
error = y_hat - y
grad  = logistic_gradient(X, y, w)

print("Forward pass:")
print(f"  scores z:  {X @ w}")
print(f"  y_hat:     {y_hat}")
print(f"  error:     {error}")
print(f"  gradient:  {grad}")
print(f"  w after 1 step (lr=0.5): {w - 0.5 * grad}")

## 2 — Gradient check vs. finite differences

In [ ]:
def loss_at(w_val):
    y_hat = sigmoid(X @ w_val)
    return cross_entropy(y, y_hat)

eps = 1e-5
fd_grad = np.array([
    (loss_at(w + eps * np.eye(2)[j]) - loss_at(w - eps * np.eye(2)[j])) / (2 * eps)
    for j in range(2)
])

analytic_grad = logistic_gradient(X, y, w)

print(f"Analytic gradient:       {analytic_grad}")
print(f"Finite-diff gradient:    {fd_grad.round(6)}")
print(f"Max absolute difference: {np.max(np.abs(analytic_grad - fd_grad)):.2e}")

assert np.allclose(analytic_grad, fd_grad, atol=1e-5), "Gradient check failed!"
print("Gradient check passed ✓")

### Validate: the analytic gradient matches finite differences

The claim is $\nabla_w L = \frac1n X^\top(\hat y - y)$. The surest check is to compare it against
a numerical gradient of the loss. They should match to many digits. We confirm.

In [ ]:
diff = np.max(np.abs(analytic_grad - fd_grad))
print(f'analytic {analytic_grad},  finite-diff {fd_grad.round(6)},  max diff {diff:.2e}')
assert diff < 1e-6, 'the analytic gradient (1/n) X^T (y_hat - y) matches finite differences'
print('\n✅ the derived gradient is correct')

## 3 — Full gradient descent on a synthetic dataset

In [ ]:
# Generate linearly separable data
n_each = 100
X_pos = np.random.randn(n_each, 2) + np.array([2, 1])
X_neg = np.random.randn(n_each, 2) + np.array([-2, -1])
X_data = np.hstack([np.ones((2*n_each, 1)), np.vstack([X_pos, X_neg])])
y_data = np.array([1]*n_each + [0]*n_each, dtype=float)

# Gradient descent
w = np.zeros(3)
lr = 0.5
losses = []

for epoch in range(500):
    y_hat = sigmoid(X_data @ w)
    losses.append(cross_entropy(y_data, y_hat))
    w -= lr * logistic_gradient(X_data, y_data, w)

acc = np.mean((sigmoid(X_data @ w) >= 0.5) == y_data)
print(f"Final loss: {losses[-1]:.4f}, accuracy: {acc:.3f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(losses, color='#6366f1')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('Loss converges to near zero')
ax1.grid(True, alpha=0.3)

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
X_grid = np.c_[np.ones(xx.ravel().shape), xx.ravel(), yy.ravel()]
Z = sigmoid(X_grid @ w).reshape(xx.shape)

ax2.contourf(xx, yy, Z, levels=30, cmap='RdYlGn', alpha=0.6)
ax2.scatter(X_pos[:, 0], X_pos[:, 1], c='#20d9d2', s=15, label='class 1')
ax2.scatter(X_neg[:, 0], X_neg[:, 1], c='#f97316', s=15, label='class 0')
# Draw the decision boundary (sigmoid=0.5 → z=0)
x_line = np.linspace(-5, 5, 100)
y_line = -(w[0] + w[1]*x_line) / (w[2] + 1e-10)
ax2.plot(x_line, y_line, 'w--', linewidth=1.5, label='decision boundary')
ax2.set_xlim(-5, 5); ax2.set_ylim(-4, 4)
ax2.set_title('Learned decision boundary')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4 — Visualize the cancellation

Plot the three factors in the chain rule separately to show that
$\partial\mathcal{L}/\partial\hat{y} \cdot \hat{y}(1-\hat{y})$ simplifies to $\hat{y}-y$.

In [ ]:
y_hat_range = np.linspace(0.01, 0.99, 500)
y_true = 1.0  # for a positive example

link1 = -(y_true / y_hat_range - (1 - y_true) / (1 - y_hat_range))  # dL/d(y_hat)
link2 = y_hat_range * (1 - y_hat_range)                                # d(y_hat)/dz = sigma'
product = link1 * link2                                                 # should = y_hat - y
expected = y_hat_range - y_true

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(y_hat_range, link1, color='#6366f1')
axes[0].set_title('Link 1: dL/d(ŷ)')
axes[0].set_xlabel('ŷ')
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_hat_range, link2, color='#f97316')
axes[1].set_title("Link 2: σ'(z) = ŷ(1-ŷ)")
axes[1].set_xlabel('ŷ')
axes[1].grid(True, alpha=0.3)

axes[2].plot(y_hat_range, product, color='#20d9d2', linewidth=2.5, label='product (links 1×2)')
axes[2].plot(y_hat_range, expected, '--', color='#ef4444', linewidth=1.5, label='ŷ − y (expected)')
axes[2].set_title('Product = ŷ − y  (exact cancellation)')
axes[2].set_xlabel('ŷ')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Validate: the chain rule cancels to $\hat y - y$

The elegance: $\frac{\partial L}{\partial z} = \frac{\partial L}{\partial \hat y}\cdot
\frac{\partial \hat y}{\partial z}$, where the first term has a $\hat y(1-\hat y)$ in the
denominator and the second *is* $\hat y(1-\hat y)$ — so they cancel, leaving $\hat y - y$. We
confirm the product equals the residual across all $\hat y$.

In [ ]:
print(f'max |link1*link2 - (y_hat - y)| = {np.abs(product - expected).max():.2e}')
assert np.allclose(product, expected), 'the chain rule collapses to y_hat - y'
print('\n✅ the sigmoid derivative cancels the cross-entropy denominator -> a clean residual gradient')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **perfect separation** | MLE at infinity; weights diverge (demo) — use L2 |
| **numerical overflow** | clip the sigmoid argument to avoid overflow |
| **log(0)** | clip predicted probabilities before the log |
| **feature scaling** | unscaled features slow gradient descent |
| **it's a linear model** | only linear decision boundaries without feature maps |

Demo: on separable data the logistic weights grow without bound.

In [ ]:
# A subtlety the clean gradient hides: on PERFECTLY SEPARABLE data the logistic MLE does not
# exist — the loss keeps decreasing as the weights grow without bound (pushing predicted
# probabilities to exactly 0/1). We train longer and confirm the weight norm keeps growing, which
# is exactly why logistic regression needs L2 regularization in practice.
def train(epochs):
    w_ = np.zeros(3)
    for _ in range(epochs):
        w_ -= 0.5 * logistic_gradient(X_data, y_data, w_)
    return w_
n_short = np.linalg.norm(train(200))
n_long = np.linalg.norm(train(2000))
print(f'||w|| after 200 epochs = {n_short:.2f};  after 2000 epochs = {n_long:.2f}')
assert n_long > 1.3 * n_short, 'on separable data the weights diverge (MLE at infinity) -> regularize'
print('\nThe residual gradient never reaches zero on separable data -> weights blow up. Add L2 (weight decay).')

## ✏️ Your turn

### Exercise 1 — Bias gradient

The gradient derivation above was for the weight vector $\mathbf{w}$.
Derive the gradient with respect to the bias $b$ (where $z_i = \mathbf{w}^T\mathbf{x}_i + b$),
implement it, and verify with finite differences.

In [ ]:
# TODO(you): derive and implement the bias gradient
# Hint: treat the bias as a weight with a constant feature x_0 = 1.
# The gradient should be: (1/n) * sum(y_hat - y)

def logistic_gradient_with_bias(X, y, w, b):
    y_hat = sigmoid(X @ w + b)
    dw = None  # TODO: (1/n) * X.T @ (y_hat - y)
    db = None  # TODO: (1/n) * sum(y_hat - y)
    return dw, db

# Verify with finite differences once you've filled in the TODO

### Exercise 2 — Softmax gradient

For 3-class logistic regression (softmax), the gradient of categorical cross-entropy
is also $\hat{\mathbf{p}} - \mathbf{y}$ (one-hot), for the same reason.
Implement softmax + categorical cross-entropy + gradient from scratch
and verify with finite differences.

<details>
<summary>Solution outline</summary>

```python
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))  # numerical stability
    return e / e.sum(axis=1, keepdims=True)

def softmax_cross_entropy_grad(X, Y_onehot, W):
    # W: (d, K), Y_onehot: (n, K)
    P = softmax(X @ W)              # (n, K) probabilities
    error = P - Y_onehot             # (n, K)
    return (1 / len(Y_onehot)) * X.T @ error   # (d, K)
```
</details>

## Key takeaways

- **The gradient is $\frac1n X^\top(\hat y - y)$** — verified against finite differences.
- **Chain-rule cancellation:** the sigmoid derivative cancels the cross-entropy denominator,
  leaving the residual (verified).
- **The bias term** is the same formula with a constant feature $x_0 = 1$.
- **Separable data breaks the MLE:** weights diverge (demo) — regularize with L2.